### Capturing My Attempts to Solve an Intriguing Programming Problem (From 2020 Onward)

Typically, you write in a blog post about a specific topic and conclude crisply. The process of writing a typical blog post may also not span more than a few days. On this blog, I intend to write blog posts like that. I even started adding dates to their titles to suggest how long they took.

This blog post, however, is unusual. I plan to narrate my (long) struggle with an intriguing problem. Capturing the writer's struggles in words, as a hypothetical, omniscient mental and physical monitoring camera might relentlessly do, is challenging. To keep things simple, I will capture my thought processes, rays of hope flashed by in the wee hours of dawns, etc., as shorter, chronological sub-posts below.

It is not an unsolved problem. People have solved it. However, until I solve it all by myself, it is unsolved for me.

I had heard the problem statement a few years ago, and I have not been able to solve it yet. I was not working solely on this problem all this time, but I can say that I have been _obsessed_ with it. No line of attack is in sight even now. I can barely withstand the emotional burden of being unable to solve this problem, despite being a CS graduate and professional programmer, but it is becoming unbearable.

I suspect it is an advanced programming problem. Advanced problems assume some background knowledge. For example, a calculus-1 problem is 'advanced' for a high schooler who hasn't yet grasped the ideas of school algebra and mathematical functions. The same problem may not be 'advanced' for a math major. 

Nobody posed this problem to me in person; I read its statement online. I can only judge its maturity level from its context: Ken Thompson, a master programmer and recipient of the [ACM Turing Award](https://en.wikipedia.org/wiki/Turing_Award), posed it during his acceptance speech of the award in 1983. He gradually solved this problem during that speech. By the time his speech ended, we had its solution (and a few other things to ponder about).

Does that make you nervous? Does the shear _intellectual weight_ of the problem poser and that occasion imply we should read "some theory" first, solve textbook exercises and problems, and only then attempt it? 

Or I can do what a great teacher of mathematics, R. L. Moore, suggested: > Read the problem statement. Try to solve it. Take your time. If you can't solve it, and a solution is available, peek at the first line of the solution. Go back to working on your own. If you are unsuccessful, repeat the process with the subsequent lines of the published solution.

That is what I decided then. Although Thompson has apparently worked out the solution during his speech, I haven't read that speech completely. I stopped reading the transcript after I read the following:

> **STAGE I**
> 
> In college, before video games, we would amuse ourselves by posing programming exercises. One of the favorites was *to write the shortest self-reproducing program*. Since this is an exercise divorced from reality, the usual vehicle was FORTRAN. Actually, FORTRAN was the language of choice for the same reason that three-legged races are popular.
>
> More precisely stated, the problem is **to write a source program that, when compiled and executed, will produce as output an exact copy of its source**. _If you have never done this, I urge you to try it on your own_. 

#### Attempt #1: Naivete (Sometime in 2023)
I have not written such a program. Thompson _urges_ us to try it on our own.

I have started seeing why this is a hard problem. 

However, my first problem-solving attempts are usually sloppy. I thought: Well, why not write C source file `one.c` with code to read a file (`one.c` itself) line by line and write the read line to the stdout?

``` c
#include <stdio.h>

int main(void) {
    FILE *f = fopen("one.c", "r");
    char buf[1024];

    if (!f) {
        perror("one.c");
        return 1;
    }

    while (fgets(buf, sizeof buf, f) != NULL) {
        fputs(buf, stdout);
    }

    fclose(f);
    return 0;
}


```
Compiling (`gcc one.c`) and executing (`./a.out`) this program on my Linux machine produces an exact copy of `one.c` to the stdout.

Of course, I cringed at my 'solution'. How could Thompson ask such a problem while giving his Turing Award lecture? Am I being a dope? Am I either ignoring or not understanding Thompson's hint ("produce the shortest _self-replicating_ program)?

When I showed this to ChatGPT, it said, "That's cheating, you are not supposed to read or write any files from or to the disk." 

It already knew about such problems. Oracular, it even told me that such a problem was called a _Quine_, named so after a logician by the same name.

I was amazed by ChatGPT's knowledge (should we say understanding?), but did not carry on with it further. I vowed to solve it _the right way_ on my own.

#### Attempt #2: Revival After Two Years (01 December 2025)
Fast forward to last night. I kept thinking about this problem once in a while in the last two years. However, didn't obsess enough over it. Looks like it doesn't go away, and I don't want to seek an answer from ChatGPT or Gemini. I don't want to read ahead in the transcript either.

My sleep was disturbed just a little around 3.40 am. I was still sleepy, but realized that this problem popped into my head. I decided to see if some fabled insight comes or the obsession hums a lullaby. Maybe a [recent blog post about compilers](./compilers-interpreters.html) was fresh in my subconscious?

Let me take a step back. Let's commit to writing C. Ken wants us to write C source program, denoted `P` that, *when compiled and executed*_, prints `P` to `stdout`. 

Consider a simple C program that includes a _call to the function that prints its arguments to `stdout`_:

``` c
#include <stdio.h>
int main(void) {
    printf("%s\n", "Hello, World!");
}
```

Compiling and executing this program produces the following to the `stdout` (ignore the initial white-space):

```
Hello, World!

```
It only prints *verbatim* to the `stdout` the second argument to `printf`. 

**How can we capture the entire file's source code within its own part?**

Here is a naive attempt:

``` c
#include <stdio.h>
int main(void) {
    printf("%s\n", "#include<stdio.h>\nint main(void) {\n    printf()\n}");
}

```

Compiling and executing this program produces the following to the `stdout` (ignore the initial white-space):

```
#include<stdio.h>
int main(void) {
    printf()
}
```
Clearly, this does not exactly match the source.

The recursive (self-referencing) nature of this program dawned on me. Even if I declare a variable to hold a string that I pass to `printf` as the second argument, the problem remains: A string can not fit into its proper substring.

Could it be related to some parsing, abstract syntax tree, or compilation magic? 

What C source line(s) might produce an executable machine code that, when run, reproduces those source lines on the `stdout`?

Am I missing some background theory of such programs? Do I head to the venerable '[Dragon Book](https://en.wikipedia.org/wiki/Compilers:_Principles,_Techniques,_and_Tools)' or the like?

Does it have to do with functions in programming languages (in particular, the `print` function), how the stored-program computer works, or what?

#### Attempt #3: Suspicion (02 December 2025)
No real progress yet. 

I drew a diagram like this (augment the web browser window, or hold the phone in portrait mode if it appears messed up):

``` text
         +------------+          +------------+               
         |            |          |            |
a.c ---> |  compiler  |--a.out-->|  computer  |---> a.c
         |            |          |            |
         +------------+          +------------+
```
In this view, if `compiler` is a mathematical function, `computer` is its inverse function!

##### Flash of Insight or Detour?
Should I suspect the compiler? Is the compiler doing something extra? Is it 

1) reading the source (`a.c`),
2) ignoring what the program is actually supposed to do when run,
3) creating the machine code to copy the source it read to the output?